# Camada Bronze
---

Os arquivos CSV já foram adicionados no Catalog, usando o próprio DataBricks. 
Foi criado um Volume (chamado Inputs) e enviado os arquivos dentro dele.

In [0]:
from pyspark.sql import functions as F

In [0]:
# Criação do DATABASE/SCHEMA
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.bronze")

In [0]:
volume_path = "/Volumes/workspace/default/inputs" # Caminho onde os arquivos CSV estão

# Será criado uma função auxiliar para a criação das tabelas Bronze. Ela será responsável por:
# 1. Ler o arquivos CSV;
# 2. Adicionar a coluna ingestion_datetime (contendo o timestamp exato do momento da inserção);
# 3. Salvar os dados em uma tabela Delta na camada Bronze.
def aux_carregar_bronze(nome_arquivo, nome_tabela):
    # Leitura do arquivos CSV
    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        # Tratamento adicional para garantir a leitura correta de campos
        # que possuem aspas internas e evitar o deslocamento das colunas.
        .option("quote", '"')
        .option("escape", '"')
        .option("unescapedQuoteHandling", "BACK_TO_DELIMITER")
        .csv(f"{volume_path}/{nome_arquivo}")
    )
    
    # Adicionar a coluna ingestion_datetime
    df = df.withColumn(
        "ingestion_datetime", F.current_timestamp()
    )

    # Gravar o DataFrame como uma tabela no formato Delta
    (
        df.write
        .format("delta")
        .mode("append")
        .saveAsTable(f"workspace.bronze.{nome_tabela}")
    )

    print(f"Tabela workspace.bronze.{nome_tabela} criada com sucesso.")

In [0]:
# Usar a função auxiliar para a criação das tabelas
aux_carregar_bronze("movies_info_TMDB_IMDB.csv", "tb_movies_info")
aux_carregar_bronze("movies_financials_IMDB_TMDB.csv", "tb_movies_financials")
aux_carregar_bronze("movies_metrics_IMDB_TMDB.csv", "tb_movies_metrics")
aux_carregar_bronze("credits_and_tags_IMDB_TMDB.csv", "tb_credits_and_tags")
aux_carregar_bronze("movies_reviews.csv", "tb_movies_reviews")

In [0]:
# Célula de verificação das tabelas criadas
spark.sql("SHOW TABLES IN workspace.bronze").show()

display(
    spark.table("workspace.bronze.tb_movies_info")
)

# Verificação da presença da coluna ingestion_datetime (ex: movies_financials)
spark.table("workspace.bronze.tb_movies_financials").printSchema()

In [0]:
# =====================================================
# Ingestão da cotação do dólar via API do Banco Central
# =====================================================
import requests
from datetime import datetime, timedelta

# Criação de widgets para permitir a definição das datas de início e fim da consulta
dbutils.widgets.text("data_inicio", "")
dbutils.widgets.text("data_fim", "")

# Definir como padrão os últimos 7 dias corridos
data_fim_padrao = datetime.today()
data_inicio_padrao = data_fim_padrao - timedelta(days=7)

data_inicio_formatada = data_inicio_padrao.strftime("%m-%d-%Y")
data_fim_formatada = data_fim_padrao.strftime("%m-%d-%Y")

# Recuperamos os valores informados nos widgets
data_inicio = dbutils.widgets.get("data_inicio")
data_fim = dbutils.widgets.get("data_fim")
# Caso os widgets estejam vazios, utilizamos os últimos 7 dias como padrão
if data_inicio == "": data_inicio = data_inicio_formatada
if data_fim == "": data_fim = data_fim_formatada

# URL da API
url = (
    "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    "CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)"
    f"?@dataInicial='{data_inicio}'"
    f"&@dataFinalCotacao='{data_fim}'"
    "&$select=dataHoraCotacao,cotacaoCompra"
    "&$format=json"
)

response = requests.get(url)
registros_cotacao = response.json()["value"]

df_cotacao = spark.createDataFrame(registros_cotacao)
# Adicionamos a data e a hora de ingestão dos dados na camada Bronze
df_cotacao = df_cotacao.withColumn(
    "ingestion_datetime",
    F.current_timestamp()
)

# Gravamos os dados em formato Delta utilizando o modo Append
df_cotacao.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("workspace.bronze.tb_cotacao_dolar")

# Exibimos a tabela criada para validar a ingestão
display(
    spark.table("workspace.bronze.tb_cotacao_dolar")
)
